# Phase 4 - Final report (Colab runner)

Runner only: mount Drive, pull the Phase 4 branch, regenerate the capstone summary from the staged
evaluation artifacts, then render the final report and the generated metrics table inline. Logic
lives in `src/` and `scripts/`, not in this notebook (P1/P2).

The metric numbers shown below are produced by `scripts/build_phase4_summary.py`; the report prose
in `reports/final_report.md` never hand-copies them.

## Boot

In [ ]:
# 1. Mount Drive so config.OUTPUT_ROOT (the staged evaluation artifacts) is available.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Get the code onto the VM and pin the Phase 4 branch.
# Repin BRANCH to 'main' once the Phase 4 PRs are merged.
import os

REPO = '/content/FinDocStructRAG'
BRANCH = 'feature/phase4-report'  # PR-B; flip to 'main' after merge

if not os.path.isdir(f'{REPO}/.git'):
    !git clone --quiet https://github.com/AD2000X/FinDocStructRAG.git {REPO}

!cd {REPO} && git fetch origin --quiet
!cd {REPO} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
!cd {REPO} && echo branch: $(git rev-parse --abbrev-ref HEAD) HEAD: $(git log --oneline -1)
%cd /content/FinDocStructRAG

In [ ]:
# 3. Make src/ importable and sanity-check the Phase 4 paths.
import importlib
import sys

sys.path.insert(0, '/content/FinDocStructRAG')
from src import config
importlib.reload(config)

print('IN_COLAB    :', config.IN_COLAB)
print('OUTPUT_ROOT :', config.OUTPUT_ROOT)
print('EVALUATION  :', config.EVALUATION)
print('LAYOUT      :', config.LAYOUT_OUTPUT)
print('reports dir :', config.ROOT / 'reports')

## Step 1 - build the capstone summary

Aggregates the per-phase artifacts on Drive into `outputs/evaluation/phase4_summary.json` and the
committed `reports/phase4_metrics.md`. Re-running is idempotent (no-drift).

In [ ]:
!python scripts/build_phase4_summary.py

## Step 2 - final report

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

display(Markdown(Path('reports/final_report.md').read_text(encoding='utf-8')))

## Step 3 - generated metrics table

The numbers cited qualitatively in the report above, generated from the summary.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

display(Markdown(Path('reports/phase4_metrics.md').read_text(encoding='utf-8')))

## Step 4 - summary JSON (artifact availability + headline)

In [ ]:
import json
from src import config

summary = json.loads((config.EVALUATION / 'phase4_summary.json').read_text(encoding='utf-8'))
print('artifact availability:')
for name, part in summary.items():
    print(f"  {name:<10} {'OK' if part.get('available') else 'MISSING'}")

f = summary.get('funsd', {})
if f.get('available'):
    h = f['headline']
    print()
    print(f"FUNSD headline ({f['primary']}): "
          f"P {h['precision']:.3f} / R {h['recall']:.3f} / F1 {h['f1']:.3f}")